# RL Trading — Comparaison d'Agents de Portefeuille
## Équipe 21 — École Polytechnique
**Antoine Delacour · Wandrille Boivin · Sylvain Dehayem · Mouhamadou Moustapha Diop**

Ce notebook présente une comparaison complète de plusieurs agents d'apprentissage par renforcement pour la gestion de portefeuille sur les actions AAPL, JPM et XOM.

---

## Section 0 : Configuration & Imports

In [1]:
# Drapeau de test rapide
QUICK_TEST = False

EXP3_EPISODES      = 2    if QUICK_TEST else 5
REINFORCE_EPISODES = 5    if QUICK_TEST else 200
PPO_EPISODES       = 2    if QUICK_TEST else 100
DDPG_TIMESTEPS     = 500  if QUICK_TEST else 20_000

print(f'QUICK_TEST={QUICK_TEST}')
print(f'EXP3_EPISODES={EXP3_EPISODES}, REINFORCE_EPISODES={REINFORCE_EPISODES}, PPO_EPISODES={PPO_EPISODES}, DDPG_TIMESTEPS={DDPG_TIMESTEPS}')

QUICK_TEST=False
EXP3_EPISODES=5, REINFORCE_EPISODES=200, PPO_EPISODES=100, DDPG_TIMESTEPS=20000


In [2]:
%matplotlib inline
import sys
import os
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import random
import torch
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('Imports OK.')

Imports OK.


In [3]:
def align_values(values, n):
    values = list(values)
    if len(values) > n:
        return values[:n]
    elif len(values) < n:
        return values + [values[-1]] * (n - len(values))
    return values

def compute_metrics(portfolio_values, daily_returns, agent_name):
    values = list(portfolio_values)
    returns = np.array(list(daily_returns))
    total_return = (values[-1] - values[0]) / values[0]
    sharpe = float((252**0.5) * returns.mean() / returns.std()) if returns.std() > 1e-9 else 0.0
    return {
        'Agent': agent_name,
        'Rendement Total (%)': round(total_return * 100, 2),
        'Sharpe Ratio': round(sharpe, 3),
        'Valeur Finale ($)': round(values[-1], 2)
    }

print('Fonctions utilitaires OK.')

Fonctions utilitaires OK.


---
## Section 1 : Environnement de Trading

**Actions :** AAPL, JPM, XOM  
**Entraînement :** 2018-01-01 à 2021-12-31  
**Test :** 2022-01-01 à 2023-12-31  
**Espace d'état :** matrice 7x3 (3 lignes covariance + 4 lignes indicateurs techniques)  
**Espace d'action :** poids de portefeuille sommant à 1

In [4]:
from environment.setup_env import build_envs

print("Construction de l'environnement...")
train_env, test_env, train_df, test_df = build_envs(reward_type='portfolio_value')

print(f'Donnees train : {train_df.shape}')
print(f'Donnees test  : {test_df.shape}')
print(f'Obs space     : {train_env.observation_space.shape}')
print(f'Action space  : {train_env.action_space.shape}')
print(f'Capital initial : ${train_env.initial_amount:,.0f}')

Construction de l'environnement...
1. Downloading data


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (4527, 8)
2. Adding technical indicators
Successfully added technical indicators
3. Computing rolling covariance
4. Splitting train / test
Train : 2019-01-03 → 2021-12-31 (2268 rows)
Test  : 2022-01-03 → 2023-12-29 (1503 rows)
5. Creating environments
Done
Donnees train : (2268, 13)
Donnees test  : (1503, 13)
Obs space     : (7, 3)
Action space  : (3,)
Capital initial : $100,000


---
## Section 2 : Agents RL

### 2.1 EXP3 — Bandit Adversarial

In [5]:
from agents.exp3 import Exp3Agent

print(f'Entrainement EXP3 ({EXP3_EPISODES} episodes)...')
exp3_agent = Exp3Agent(n_arms=3, gamma=0.1)
exp3_agent.train(train_env, n_episodes=EXP3_EPISODES)

print('Test EXP3...')
exp3_results = exp3_agent.test(test_env)

print(f"[EXP3] Rendement total : {exp3_results['total_return']*100:.2f}%")
print(f"[EXP3] Sharpe ratio    : {exp3_results['sharpe']:.3f}")
print(f"[EXP3] Valeur finale   : ${exp3_results['portfolio_values'][-1]:,.2f}")

Entrainement EXP3 (5 episodes)...
begin_total_asset:100000
end_total_asset:245573.33453416394
Sharpe:  1.019862959097818
[Exp3] Episode 1/5 — final portfolio value : 245,573.33 — weights : [818036.316  571502.0953 165246.6891]
begin_total_asset:100000
end_total_asset:312700.5247545379
Sharpe:  1.2407668991299123
[Exp3] Episode 2/5 — final portfolio value : 312,700.52 — weights : [3.94734315e+11 2.60392089e+11 1.89940638e+11]
begin_total_asset:100000
end_total_asset:479691.02604466176
Sharpe:  1.7077472335532842
[Exp3] Episode 3/5 — final portfolio value : 479,691.03 — weights : [3.97996295e+17 1.30089333e+16 1.46341981e+17]
begin_total_asset:100000
end_total_asset:580221.6729946461
Sharpe:  1.858185467500083
[Exp3] Episode 4/5 — final portfolio value : 580,221.67 — weights : [2.57417794e+23 2.06737610e+22 7.78208741e+22]
begin_total_asset:100000
end_total_asset:461426.1172609749
Sharpe:  1.6757259256747865
[Exp3] Episode 5/5 — final portfolio value : 461,426.12 — weights : [1.20791619e

### 2.2 REINFORCE — Gradient de Politique

In [6]:
from agents.reinforce import ReinforceAgent
from agents.DRL import DRLAgent

train_env_r, test_env_r, _, _ = build_envs(reward_type='portfolio_value')

print(f'Entrainement REINFORCE ({REINFORCE_EPISODES} episodes)...')
reinforce_agent = ReinforceAgent(obs_dim=21, action_dim=3)
reinforce_agent.train(train_env_r, n_episodes=REINFORCE_EPISODES, print_every=max(1, REINFORCE_EPISODES // 5))

print('Evaluation REINFORCE...')
total_reward_r, actions_r = reinforce_agent.evaluate(test_env_r)

reinforce_values = list(test_env_r.asset_memory)
reinforce_returns = list(test_env_r.portfolio_return_memory)
reinforce_metrics = compute_metrics(reinforce_values, reinforce_returns, 'REINFORCE')

print(f"[REINFORCE] Rendement total : {reinforce_metrics['Rendement Total (%)']:.2f}%")
print(f"[REINFORCE] Sharpe ratio    : {reinforce_metrics['Sharpe Ratio']:.3f}")
print(f"[REINFORCE] Valeur finale   : ${reinforce_metrics['Valeur Finale ($)']:,.2f}")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

1. Downloading data
Shape of DataFrame:  (4527, 8)
2. Adding technical indicators


Successfully added technical indicators
3. Computing rolling covariance
4. Splitting train / test
Train : 2019-01-03 → 2021-12-31 (2268 rows)
Test  : 2022-01-03 → 2023-12-29 (1503 rows)
5. Creating environments
Done
Entrainement REINFORCE (200 episodes)...
begin_total_asset:100000
end_total_asset:249527.16923437832
Sharpe:  1.1815029544281186
begin_total_asset:100000
end_total_asset:227750.24097406198
Sharpe:  1.0759123482739568
begin_total_asset:100000
end_total_asset:224678.41133672663
Sharpe:  1.0632394565475016
begin_total_asset:100000
end_total_asset:223020.2843414879
Sharpe:  1.0539496948107965
begin_total_asset:100000
end_total_asset:227381.13278340653
Sharpe:  1.0827099414523274
begin_total_asset:100000
end_total_asset:227465.8185333707
Sharpe:  1.079362669453629
begin_total_asset:100000
end_total_asset:220561.00213644665
Sharpe:  1.0342038264201787
begin_total_asset:100000
end_total_asset:224881.36626142
Sharpe:  1.0698908477514586
begin_total_asset:100000
end_total_asset:2291

### 2.3 PPO — Proximal Policy Optimization

In [7]:
from agents.ppo import PPO
from agents.architecture import SimplePortfolioMLP

train_env_ppo, test_env_ppo, _, _ = build_envs(reward_type='portfolio_value')

print(f'Entrainement PPO ({PPO_EPISODES} episodes)...')
ppo_agent = PPO(
    env=train_env_ppo,
    policy=SimplePortfolioMLP,
    policy_kwargs={
        'input_shape': (7, 3),
        'portfolio_size': 3,
        'hidden_dim': 64,
    },
    batch_size=64,
    lr=1e-3,
    gamma=0.99,
    clip_epsilon=0.2,
    update_epochs=5,
    dirichlet_scale=50.0,
    verbose=1,
    seed=42,
)
ppo_agent.train(episodes=PPO_EPISODES)

print('Test PPO...')
ppo_agent.test(test_env_ppo)

ppo_values = list(test_env_ppo.asset_memory)
ppo_returns = list(test_env_ppo.portfolio_return_memory)
ppo_metrics = compute_metrics(ppo_values, ppo_returns, 'PPO')

print(f"[PPO] Rendement total : {ppo_metrics['Rendement Total (%)']:.2f}%")
print(f"[PPO] Sharpe ratio    : {ppo_metrics['Sharpe Ratio']:.3f}")
print(f"[PPO] Valeur finale   : ${ppo_metrics['Valeur Finale ($)']:,.2f}")

1. Downloading data


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (4527, 8)
2. Adding technical indicators


Successfully added technical indicators
3. Computing rolling covariance
4. Splitting train / test
Train : 2019-01-03 → 2021-12-31 (2268 rows)
Test  : 2022-01-03 → 2023-12-29 (1503 rows)
5. Creating environments
Done
Entrainement PPO (100 episodes)...


Training PPO:   1%|          | 1/100 [00:00<00:27,  3.58it/s, reward=0.6546, loss=0.196316, pv=227615.92, steps=756]

...

begin_total_asset:100000
end_total_asset:311544.2972933196
Sharpe:  1.472365494451459


Training PPO:  99%|█████████▉| 99/100 [00:26<00:00,  3.85it/s, reward=1.6402, loss=0.929055, pv=312034.28, steps=756]

begin_total_asset:100000
end_total_asset:312034.27668374544
Sharpe:  1.4742665154719259


Training PPO: 100%|██████████| 100/100 [00:26<00:00,  3.77it/s, reward=1.6389, loss=0.930989, pv=311856.19, steps=756]

begin_total_asset:100000
end_total_asset:311856.19100003754
Sharpe:  1.4735522175391684
Test PPO...
begin_total_asset:100000
end_total_asset:122746.7951149283
Sharpe:  0.5701185405057632
[PPO] Rendement total : 22.75%
[PPO] Sharpe ratio    : 0.571
[PPO] Valeur finale   : $122,746.80


---
## Section 3 : Baselines

### 3.1 Buy & Hold

In [8]:
from baselines.baselines import BuyAndHoldBaseline

_, test_env_bnh, _, _ = build_envs(reward_type='portfolio_value')

print('Test Buy & Hold...')
bnh = BuyAndHoldBaseline()
bnh_results = bnh.test(test_env_bnh)

bnh_metrics = compute_metrics(bnh_results['portfolio_values'], bnh_results['daily_returns'], 'Buy & Hold')
print(f"[Buy & Hold] Rendement total : {bnh_metrics['Rendement Total (%)']:.2f}%")
print(f"[Buy & Hold] Sharpe ratio    : {bnh_metrics['Sharpe Ratio']:.3f}")
print(f"[Buy & Hold] Valeur finale   : ${bnh_metrics['Valeur Finale ($)']:,.2f}")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

1. Downloading data
Shape of DataFrame:  (4527, 8)
2. Adding technical indicators


Successfully added technical indicators
3. Computing rolling covariance
4. Splitting train / test
Train : 2019-01-03 → 2021-12-31 (2268 rows)
Test  : 2022-01-03 → 2023-12-29 (1503 rows)
5. Creating environments
Done
Test Buy & Hold...
begin_total_asset:100000
end_total_asset:131139.18835714893
Sharpe:  0.7516838757618384
[Buy & Hold] Rendement total : 31.14%
[Buy & Hold] Sharpe ratio    : 0.752
[Buy & Hold] Valeur finale   : $131,139.19


### 3.2 DDPG — Deep Deterministic Policy Gradient

In [9]:
from baselines.baselines import DDPGBaseline

train_env_ddpg, test_env_ddpg, _, _ = build_envs(reward_type='portfolio_value')

print(f'Entrainement DDPG ({DDPG_TIMESTEPS} timesteps)...')
ddpg = DDPGBaseline()
ddpg.train(train_env_ddpg, timesteps=DDPG_TIMESTEPS)

print('Test DDPG...')
ddpg_results = ddpg.test(test_env_ddpg)

ddpg_metrics = compute_metrics(ddpg_results['portfolio_values'], ddpg_results['daily_returns'], 'DDPG')
print(f"[DDPG] Rendement total : {ddpg_metrics['Rendement Total (%)']:.2f}%")
print(f"[DDPG] Sharpe ratio    : {ddpg_metrics['Sharpe Ratio']:.3f}")
print(f"[DDPG] Valeur finale   : ${ddpg_metrics['Valeur Finale ($)']:,.2f}")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

1. Downloading data
Shape of DataFrame:  (4527, 8)
2. Adding technical indicators


Successfully added technical indicators
3. Computing rolling covariance
4. Splitting train / test
Train : 2019-01-03 → 2021-12-31 (2268 rows)
Test  : 2022-01-03 → 2023-12-29 (1503 rows)
5. Creating environments
Done
Entrainement DDPG (20000 timesteps)...
begin_total_asset:100000
end_total_asset:266592.7648464426
Sharpe:  1.309187600116854
begin_total_asset:100000
end_total_asset:309319.99205496116
Sharpe:  1.467447607753832
begin_total_asset:100000
end_total_asset:307347.549772829
Sharpe:  1.4583845863906904
begin_total_asset:100000
end_total_asset:304057.2583567727
Sharpe:  1.4471834398051677
begin_total_asset:100000
end_total_asset:309516.34141396004
Sharpe:  1.46862241586101
begin_total_asset:100000
end_total_asset:307916.68339059997
Sharpe:  1.4597820167165059
begin_total_asset:100000
end_total_asset:309105.79823633
Sharpe:  1.4647404936616772
begin_total_asset:100000
end_total_asset:306059.18447473686
Sharpe:  1.453645989768028
begin_total_asset:100000
end_total_asset:307276.46035

---
## Section 4 : Resultats & Comparaison

### 4.1 Tableau des metriques

In [10]:
exp3_metrics = compute_metrics(exp3_results['portfolio_values'], exp3_results['daily_returns'], 'EXP3')

all_metrics = [exp3_metrics, reinforce_metrics, ppo_metrics, bnh_metrics, ddpg_metrics]
metrics_df = pd.DataFrame(all_metrics).sort_values('Rendement Total (%)', ascending=False).reset_index(drop=True)

print('=' * 70)
print('COMPARAISON DES AGENTS — Periode de test 2022-2023')
print('=' * 70)
print(metrics_df.to_string(index=False))
print('=' * 70)

COMPARAISON DES AGENTS — Periode de test 2022-2023
     Agent  Rendement Total (%)  Sharpe Ratio  Valeur Finale ($)
 REINFORCE                35.09         0.798          135093.40
Buy & Hold                31.14         0.752          131139.19
      DDPG                22.78         0.571          122778.60
       PPO                22.75         0.571          122746.80
      EXP3                19.39         0.460          119386.53


### 4.2 Courbes de valeur de portefeuille

In [11]:
test_dates = pd.to_datetime(test_env_bnh.date_memory)
n_dates = len(test_dates)

agent_series = {
    'EXP3':       align_values(exp3_results['portfolio_values'], n_dates),
    'REINFORCE':  align_values(reinforce_values, n_dates),
    'PPO':        align_values(ppo_values, n_dates),
    'Buy & Hold': align_values(bnh_results['portfolio_values'], n_dates),
    'DDPG':       align_values(ddpg_results['portfolio_values'], n_dates),
}

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

fig, ax = plt.subplots(figsize=(14, 7))
for (agent_name, values), color in zip(agent_series.items(), colors):
    ax.plot(test_dates, values, label=agent_name, color=color, linewidth=2)

ax.set_title('Valeur du Portefeuille — Periode de Test (2022-2023)', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Valeur du Portefeuille ($)', fontsize=13)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=12, loc='best')
ax.grid(True, alpha=0.3)
ax.set_xlim(test_dates.min(), test_dates.max())

plt.tight_layout()
plt.show()


### 4.3 Allocations de portefeuille par agent

In [12]:

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

# ── EXP3 allocations ──────────────────────────────────────────────────────────
# exp3_results["actions"] contains raw logit vectors like [10., 0., 0.]
# Apply softmax to get actual portfolio weights
exp3_alloc = np.array([softmax(a) for a in exp3_results["actions"]])

# ── REINFORCE allocations ─────────────────────────────────────────────────────
# actions_r is the second return value from reinforce_agent.evaluate(test_env_r)
# It's a list of weight arrays already normalized (Dirichlet mean)
reinforce_alloc = np.array(actions_r)

# ── PPO allocations ────────────────────────────────────────────────────────────
# test_env_ppo.save_action_memory() returns a DataFrame (n_steps+1, 3) with weights
ppo_alloc_df = test_env_ppo.save_action_memory()
ppo_alloc = ppo_alloc_df.values  # shape (n_steps+1, 3)

# ── Buy & Hold allocations ────────────────────────────────────────────────────
# Constant equal-weight allocation
test_dates = pd.to_datetime(test_env_bnh.date_memory)
bnh_alloc = np.full((len(test_dates), 3), 1.0 / 3.0)

# ── DDPG allocations ──────────────────────────────────────────────────────────
# test_env_ddpg.save_action_memory() returns a DataFrame with weights
ddpg_alloc_df = test_env_ddpg.save_action_memory()
ddpg_alloc = ddpg_alloc_df.values  # shape (n_steps+1, 3)

stock_names = ['AAPL', 'JPM', 'XOM']
alloc_colors = ['#3498db', '#e74c3c', '#2ecc71']

# ── Align all allocations to the same date range ─────────────────────────────
n_dates = len(test_dates)

def _align_alloc(arr, n):
    """Trim or pad allocation array to exactly n rows."""
    arr = np.array(arr)
    if len(arr) >= n:
        return arr[:n]
    pad = np.tile(arr[-1], (n - len(arr), 1))
    return np.vstack([arr, pad])

alloc_dict = {
    'EXP3':       _align_alloc(exp3_alloc,     n_dates),
    'REINFORCE':  _align_alloc(reinforce_alloc, n_dates),
    'PPO':        _align_alloc(ppo_alloc,       n_dates),
    'Buy & Hold': _align_alloc(bnh_alloc,       n_dates),
    'DDPG':       _align_alloc(ddpg_alloc,      n_dates),
}

# ── Plot: 2×3 grid (5 agents, last cell empty) ────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes_flat = axes.flatten()

agent_order = ['EXP3', 'REINFORCE', 'PPO', 'Buy & Hold', 'DDPG']

for idx, agent_name in enumerate(agent_order):
    ax = axes_flat[idx]
    alloc = alloc_dict[agent_name]   # shape (n_dates, 3)

    # Normalise each row so weights sum to 1 (safety guard)
    row_sums = alloc.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums < 1e-9, 1.0, row_sums)
    alloc_norm = alloc / row_sums

    ax.stackplot(
        test_dates,
        alloc_norm[:, 0],
        alloc_norm[:, 1],
        alloc_norm[:, 2],
        labels=stock_names,
        colors=alloc_colors,
        alpha=0.85,
    )
    ax.set_title(agent_name, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_ylabel('Poids', fontsize=10)
    ax.set_xlim(test_dates.min(), test_dates.max())
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.grid(True, alpha=0.25)

    if idx == 0:
        ax.legend(stock_names, loc='upper right', fontsize=9)

# Hide the unused 6th subplot
axes_flat[-1].set_visible(False)

fig.suptitle(
    "Stratégies d'allocation par agent — Période de test 2022–2023",
    fontsize=15, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()


### 4.4 Ratios de Sharpe comparés

In [13]:
fig, ax = plt.subplots(figsize=(10, 5))

agents_names = metrics_df['Agent'].tolist()
sharpe_vals = metrics_df['Sharpe Ratio'].tolist()
bar_colors = ['#2ecc71' if s > 0 else '#e74c3c' for s in sharpe_vals]

bars = ax.barh(agents_names, sharpe_vals, color=bar_colors, edgecolor='white', height=0.6)

for bar, val in zip(bars, sharpe_vals):
    x_pos = val + 0.01 if val >= 0 else val - 0.01
    ha = 'left' if val >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', ha=ha, fontsize=11, fontweight='bold')

ax.axvline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax.set_title('Ratios de Sharpe Annualises — Periode de Test', fontsize=15, fontweight='bold', pad=12)
ax.set_xlabel('Sharpe Ratio (annualise, sqrt(252))', fontsize=12)
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()


---
## Section 5 : Ablation Study — Fonction de Récompense

On réentraîne les 3 agents (EXP3, REINFORCE, PPO) avec chaque fonction de récompense pour mesurer l'impact du choix du signal de récompense sur les performances.

In [14]:
from application.run_comparison import run_comparison

ABL_EXP3      = 2  if QUICK_TEST else 5
ABL_REINFORCE = 2  if QUICK_TEST else 30
ABL_PPO       = 2  if QUICK_TEST else 30

reward_types = ["portfolio_value", "portfolio_return", "log_return"]
ablation_summaries    = {}
ablation_comparisons  = {}  # NEW: store time-series per reward type

for reward_type in reward_types:
    print(f"\n{'='*60}")
    print(f"Ablation : reward_type = '{reward_type}'")
    print(f"{'='*60}")
    
    artifacts = run_comparison(
        if_using_exp3=True,
        if_using_reinforce=True,
        if_using_ppo=True,
        if_using_policy_gradient=False,
        if_using_dqn=False,
        reward_type=reward_type,
        exp3_train_episodes=ABL_EXP3,
        reinforce_train_episodes=ABL_REINFORCE,
        ppo_train_episodes=ABL_PPO,
        print_test_results=True,
    )
    ablation_summaries[reward_type]   = artifacts["summary"]
    ablation_comparisons[reward_type] = artifacts["comparison"]  # NEW

print("\nAblation terminée.")

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Ablation : reward_type = 'portfolio_value'
1. Downloading data
Shape of DataFrame:  (4527, 8)
2. Adding technical indicators


Successfully added technical indicators
3. Computing rolling covariance
4. Splitting train / test
Train : 2019-01-03 → 2021-12-31 (2268 rows)
Test  : 2022-01-03 → 2023-12-29 (1503 rows)
5. Creating environments
Done
<class 'environment.portfolio_env.StockPortfolioEnv'>
[run] PPO
[train] PPO episodes=30


Training PPO:   3%|▎         | 1/30 [00:00<00:11,  2.42it/s, reward=0.0508, loss=0.091624, pv=173961.40, steps=756]

...

Training PPO:  97%|█████████▋| 29/30 [00:07<00:00,  3.72it/s, reward=1.6379, loss=0.911900, pv=311741.49, steps=756]

begin_total_asset:100000
end_total_asset:311741.4943591342
Sharpe:  1.4734807690653158


Training PPO: 100%|██████████| 30/30 [00:08<00:00,  3.69it/s, reward=1.6377, loss=0.914483, pv=311710.08, steps=756]

begin_total_asset:100000
end_total_asset:311710.0821826636
Sharpe:  1.473318255967266
begin_total_asset:100000
end_total_asset:122771.77283099464
Sharpe:  0.5705692048329429
[run] Exp3
[train] Exp3 episodes=5


begin_total_asset:100000
end_total_asset:207755.6647721758
Sharpe:  0.860614241413587
[Exp3] Episode 1/5 — final portfolio value : 0.01 — weights : [445.0377 951.1377 909.9526]
begin_total_asset:100000
end_total_asset:218626.2891477277
Sharpe:  0.9217523165647664
[Exp3] Episode 2/5 — final portfolio value : -0.00 — weights : [297736.93    53966.6657 885020.1053]
begin_total_asset:100000
end_total_asset:157537.80931309646
Sharpe:  0.6146040985761472
[Exp3] Episode 3/5 — final portfolio value : 0.01 — weights : [1.02233804e+08 3.51465386e+07 6.01986836e+08]
begin_total_asset:100000
end_total_asset:166041.45148354903
Sharpe:  0.6400047166645711
[Exp3] Episode 4/5 — final portfolio value : 0.01 — weights : [2.39950762e+09 1.05830959e+11 1.00382810e+12]
begin_total_asset:100000
end_total_asset:134363.01290036197
Sharpe:  0.4505560633626625
[Exp3] Episode 5/5 — final portfolio value : 0.01 — weights : [2.83795216e+12 4.31089355e+13 7.14172099e+14]
begin_total_asset:100000
end_total_asset:170

In [15]:
reward_labels = {
    "portfolio_value":   "Portfolio Value (valeur brute)",
    "portfolio_return":  "Portfolio Return (rendement journalier)",
    "log_return":        "Log Return (log(1 + rendement))",
}

for reward_type, summary_df in ablation_summaries.items():
    print(f"\n{'─'*55}")
    print(f"  Reward : {reward_labels[reward_type]}")
    print(f"{'─'*55}")
    table = summary_df[["model", "total_return", "sharpe", "final_portfolio_value"]].copy()
    table.columns = ["Agent", "Rendement Total", "Sharpe", "Valeur Finale ($)"]
    table["Rendement Total"] = (table["Rendement Total"] * 100).round(2).astype(str) + "%"
    table["Sharpe"] = table["Sharpe"].round(3)
    table["Valeur Finale ($)"] = table["Valeur Finale ($)"].round(2)
    print(table.to_string(index=False))


───────────────────────────────────────────────────────
  Reward : Portfolio Value (valeur brute)
───────────────────────────────────────────────────────
    Agent Rendement Total  Sharpe  Valeur Finale ($)
      ppo          44.69%   0.933          144691.08
reinforce          36.81%   0.838          136810.82
     exp3          19.39%   0.459          119386.53

───────────────────────────────────────────────────────
  Reward : Portfolio Return (rendement journalier)
───────────────────────────────────────────────────────
    Agent Rendement Total  Sharpe  Valeur Finale ($)
     exp3          70.45%   1.029          170451.20
reinforce          35.11%   0.803          135106.09
      ppo          22.77%   0.571          122768.05

───────────────────────────────────────────────────────
  Reward : Log Return (log(1 + rendement))
───────────────────────────────────────────────────────
    Agent Rendement Total  Sharpe  Valeur Finale ($)
     exp3          70.45%   1.029          17045

In [16]:
import matplotlib.pyplot as plt
import numpy as np

reward_labels = {
    "portfolio_value":  "Portfolio Value",
    "portfolio_return": "Portfolio Return",
    "log_return":       "Log Return",
}
agent_colors = {"exp3": "#e74c3c", "reinforce": "#3498db", "ppo": "#2ecc71"}

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)

for idx, (reward_type, summary_df) in enumerate(ablation_summaries.items()):
    ax = axes[idx]
    agents  = summary_df["model"].tolist()
    returns = (summary_df["total_return"] * 100).tolist()
    colors  = [agent_colors.get(a, "#95a5a6") for a in agents]
    
    bars = ax.bar(agents, returns, color=colors, edgecolor="white", linewidth=0.8)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.set_title(f"reward = {reward_labels[reward_type]}", fontsize=12, fontweight="bold")
    ax.set_ylabel("Rendement Total (%)" if idx == 0 else "")
    ax.set_xlabel("Agent")
    
    for bar, val in zip(bars, returns):
        offset = 0.5 if val >= 0 else -1.5
        va = "bottom" if val >= 0 else "top"
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + offset,
                f"{val:.1f}%", ha="center", va=va, fontsize=10, fontweight="bold")

fig.suptitle("Ablation — Rendement Final par Reward Function × Agent\nPériode de test : 2022–2023",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [17]:
import pandas as pd
import matplotlib.ticker as mticker

agent_colors_ts = {"exp3": "#e74c3c", "reinforce": "#3498db", "ppo": "#2ecc71"}
agent_labels    = {"exp3": "EXP3", "reinforce": "REINFORCE", "ppo": "PPO"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

for idx, (reward_type, comp_df) in enumerate(ablation_comparisons.items()):
    ax = axes[idx]
    comp_df = comp_df.copy()
    comp_df["date"] = pd.to_datetime(comp_df["date"])
    
    for agent_name, agent_df in comp_df.groupby("model"):
        agent_df = agent_df.sort_values("date")
        color = agent_colors_ts.get(agent_name, "#95a5a6")
        label = agent_labels.get(agent_name, agent_name)
        ax.plot(agent_df["date"], agent_df["portfolio_value"],
                label=label, color=color, linewidth=2)
    
    ax.set_title(f"reward = {reward_labels[reward_type]}", fontsize=12, fontweight="bold")
    ax.set_ylabel("Valeur ($)" if idx == 0 else "")
    ax.set_xlabel("Date")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=30, labelsize=8)

fig.suptitle("Ablation — Évolution du Portefeuille selon la Reward Function\nPériode de test : 2022–2023",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


---
## Section 6 : Conclusion

### Resume des resultats

Ce notebook a compare cinq strategies de gestion de portefeuille sur les actions AAPL, JPM et XOM (2022-2023) :

**Agents RL :**
- **EXP3** : Bandit adversarial sans etat. Robuste aux marches non-stationnaires mais ignore l'information d'etat.
- **REINFORCE** : Politique de Dirichlet avec gradient Monte Carlo. Exploite l'espace d'etat mais souffre d'une haute variance.
- **PPO** : Version stabilisee de REINFORCE avec clipping. Meilleure stabilite d'entrainement.

**Baselines :**
- **Buy & Hold** : Reference passive, allocation equi-ponderee fixe.
- **DDPG** : Actor-critic hors-politique, plus complexe mais pas necessairement superieur.

### Points cles

1. La **fonction de recompense** influence les performances (Section 5). Le choix entre `portfolio_value`, `portfolio_return` et `log_return` affecte les comportements d'exploration.

2. Les **marches 2022-2023** tres volatils (hausse des taux, inflation) testent la robustesse des agents.

3. Le **nombre d'episodes** est determinant : les resultats en mode QUICK_TEST sont limites par le sous-apprentissage.

4. **EXP3** offre un excellent rapport complexite/performance pour les marches adversariaux.

In [4]:
import time
from agents.exp3 import Exp3Agent

agent = Exp3Agent(n_arms=3, gamma=0.1, seed=42)
start = time.perf_counter()
for _ in range(1000):
    arm, probs = agent.select_arm()
end = time.perf_counter()
print(f"Mean inference latency: {(end-start)/1000*1000:.4f} ms")

Mean inference latency: 0.0132 ms
